<a href="https://colab.research.google.com/github/Elazar-segal/colab/blob/main/%D7%9E%D7%90%D7%9E%D7%9F_%D7%9B%D7%95%D7%A9%D7%A8_%D7%97%D7%9B%D7%9D_%D7%92%D7%A8%D7%A1%D7%AA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
========================================================================
פרויקט מאמן כושר חכם (Smart Fitness Coach) - גרסה מותאמת ל-Google Colab
========================================================================

הסבר על הקוד והפעלתו:
--------------------
סביבת Google Colab פועלת על שרתים בענן ולא על המחשב האישי שלכם.
לכן, אי אפשר להפעיל את מצלמת הרשת שלכם בזמן אמת (Live) או לפתוח חלונות קופצים (cv2.imshow).
הפתרון: הקוד הזה מקבל סרטון וידאו שהקלטתם מראש, מנתח אותו פריים אחר פריים,
ושומר סרטון חדש שבו מצויר השלד ומופיע מונה החזרות.

איך מפעילים את הקוד ב-Colab?
----------------------------
1. צלמו את עצמכם בטלפון מבצעים שכיבות סמיכה (רצוי צילום מהצד שבו רואים את כל הגוף).
2. העבירו את הסרטון למחשב, ושנו את השם שלו ל: my_exercise.mp4
3. בתפריט השמאלי ב-Colab (סמל של תיקייה 📁), לחצו על סמל ה"העלאה" (Upload) והעלו את הסרטון.
4. ודאו שהתקנתם את הספריות הנדרשות בתא קוד נפרד מעל הקוד הזה:
   !pip install mediapipe opencv-python
5. הריצו את תא הקוד הנוכחי (לחיצה על כפתור ה-Play).
6. המתינו עד שהקוד יסיים (יופיע כיתוב "העיבוד הסתיים!").
7. בתפריט התיקיות משמאל יופיע קובץ חדש בשם: output_result.mp4.
   לחצו עליו מקש ימני -> Download (הורד) וצפו בתוצאה המדהימה שלכם!
========================================================================
"""

import cv2
import mediapipe as mp
import numpy as np

# הגדרת אובייקטים של MediaPipe לזיהוי גוף (Pose) וציור
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

def calculate_angle(a, b, c):
    """
    פונקציה מתמטית לחישוב הזווית בין 3 נקודות (כתף, מרפק, כף יד).
    """
    a = np.array(a) # נקודה 1 (למשל כתף)
    b = np.array(b) # קודקוד הזווית (למשל מרפק)
    c = np.array(c) # נקודה 3 (למשל שורש כף יד)

    # שימוש בטריגונומטריה לחילוץ הזווית
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

# ==================== אתחול המערכת ====================
counter = 0     # מונה חזרות
stage = None    # משתנה שומר מצב (State): "up" או "down"

# שם הקובץ שהעליתם ל-Colab (חובה לוודא שזה השם!)
input_video_path = 'my_exercise.mp4'
cap = cv2.VideoCapture(input_video_path)

# בדיקה אם הסרטון נטען בהצלחה
if not cap.isOpened():
    print("שגיאה: לא הצלחתי למצוא את הקובץ my_exercise.mp4.")
    print("אנא ודאו שהעליתם אותו לתיקיית הקבצים בצד שמאל וששמו מדויק.")
    exit()

# הגדרות לשמירת הסרטון החדש שייווצר
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# הגדרת "הסרטון היוצא" (התוצאה)
output_video_path = 'output_result.mp4'
out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

print("מתחיל בעיבוד הסרטון... זה עשוי לקחת כמה דקות, אנא המתינו.")

# ==================== הלולאה המרכזית ====================
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # אם ret הוא False, סימן שהגענו לסוף הסרטון
        if not ret:
            break

        # המרת צבעים למבנה ש-MediaPipe יודע לקרוא
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # שליחת התמונה למודל הבינה המלאכותית לזיהוי המפרקים
        results = pose.process(image)

        # החזרת הצבעים למצב רגיל כדי שנוכל לצייר עליהם
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # מנסים לחלץ את הנקודות ולחשב זוויות (בתוך try כי אולי אין אדם בפריים)
        try:
            landmarks = results.pose_landmarks.landmark

            # 1. חילוץ המיקום (x, y) של צד שמאל של הגוף (כתף, מרפק, כף יד)
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
            wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

            # 2. חישוב הזווית הנוכחית
            angle = calculate_angle(shoulder, elbow, wrist)

            # ציור הזווית על המסך (בסמוך למרפק)
            cv2.putText(image, str(int(angle)),
                           tuple(np.multiply(elbow, [width, height]).astype(int)),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            # 3. מכונת המצבים (State Machine) - הלוגיקה של הספירה
            # אם המרפק ישר יחסית (זווית גדולה מ-160)
            if angle > 160:
                stage = "up"
            # אם המרפק כפוף (זווית קטנה מ-90) וקודם היינו למעלה - הושלמה ירידה!
            if angle < 90 and stage == 'up':
                stage = "down"
                counter += 1 # מוסיפים 1 למונה
                print(f"זוהתה חזרה! סך הכל עד כה: {counter}")

        except:
            # אם לא זוהה גוף, נתעלם ונמשיך לפריים הבא
            pass

        # 4. ציור ממשק המשתמש (GUI) על המסך
        cv2.rectangle(image, (0,0), (225,120), (245,117,16), -1) # קופסת רקע

        # טקסט: REPS (חזרות)
        cv2.putText(image, 'REPS', (15,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter), (10,100), cv2.FONT_HERSHEY_SIMPLEX, 2.5, (255,255,255), 4, cv2.LINE_AA)

        # טקסט: STAGE (מצב נוכחי)
        cv2.putText(image, 'STAGE', (100,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 1, cv2.LINE_AA)
        if stage is not None:
            cv2.putText(image, stage, (100,80), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255,255,255), 2, cv2.LINE_AA)

        # 5. ציור השלד עצמו על גבי האדם
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                    mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                                    mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))

        # 6. שמירת התמונה המוכנה אל תוך הסרטון החדש
        out.write(image)

# ==================== סיום וניקוי ====================
cap.release()
out.release()
print("=====================================================")
print("העיבוד הסתיים בהצלחה!")
print("סך הכל חזרות שנספרו: ", counter)
print("קובץ חדש בשם 'output_result.mp4' נוצר בתיקיית הקבצים.")
print("אתם יכולים להוריד אותו ולצפות בו במחשב שלכם.")
print("=====================================================")